# Multi-head attention from nn.MultiheadAttention's packed weights

Checkpoints often store attention as one **packed** input projection, as `torch.nn.MultiheadAttention` does. Loading such weights into your own kernel (or the other way round) means knowing the exact layout. Re-implement the module's forward pass from its parameters.

Write `mha_packed(x, in_w, in_b, out_w, out_b, n_heads, key_padding_mask=None, causal=False)`.

## Parameters (PyTorch `nn.Linear` convention: $y = x W^\top + b$)
- `x`: `(B, T, D)` float64 (batch first).
- `in_w`: `(3D, D)`, `in_b`: `(3D,)`. Rows `0:D` are the query projection, `D:2D` the key projection, `2D:3D` the value projection (same split for the bias).
- `out_w`: `(D, D)`, `out_b`: `(D,)`.
- `n_heads` $= H$ divides $D$; head $h$ owns features $h\,d_h$ to $(h+1)\,d_h - 1$ with $d_h = D/H$.
- `key_padding_mask`: optional bool `(B, T)`. **`True` marks padding**, which no query may attend to (PyTorch's convention).
- `causal`: if true, query $i$ may only attend to keys $j \le i$.

Tests guarantee every query can see at least one key.

## Output
A tuple `(out, attn)`:
- `out`: `(B, T, D)`, the module output: per-head $\mathrm{softmax}(Q_h K_h^\top / \sqrt{d_h})\,V_h$, heads concatenated in order, then the output projection.
- `attn`: `(B, T, T)`, the attention weights **averaged over heads** (what the module returns with `need_weights=True`).

No Python loops over batch, heads or positions. You can check your function by loading the same tensors into `nn.MultiheadAttention(D, H, batch_first=True, dtype=torch.float64)`.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def mha_packed(x, in_w, in_b, out_w, out_b, n_heads, key_padding_mask=None, causal=False):
    """Return (out (B,T,D), head-averaged attention weights (B,T,T))."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[1.0, 0.0], [0.0, 1.0]]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, 1], "e": {"__tup__": [{"__nd__": [[[0.6697615493266569, 0.33023845067334306], [0.33023845067334306, 0.6697615493266569]]], "dtype": "float64"}, {"__nd__": [[[0.6697615493266569, 0.33023845067334306], [0.33023845067334306, 0.6697615493266569]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[1.0, 0.0], [0.0, 1.0]]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, 2, null, true], "e": {"__tup__": [{"__nd__": [[[1.0, 0.0], [0.5, 0.7310585786300049]]], "dtype": "float64"}, {"__nd__": [[[1.0, 0.0], [0.3844707106849976, 0.6155292893150024]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-0.5749490057210601, 0.7035450933309114, 1.5793726520932216, 0.4212106442782976, -0.7461519975907094, 0.2971315096374385, -0.01661920098083325, -0.2037407223881694], [-0.7344710985474203, 0.38725975594300555, 0.3078796263658112, -0.09298412757635025, -0.2216879397006701, -1.284916570539673, -0.48617566714143334, 1.2064496713466424], [-0.19055825686602823, -1.4396973168381253, 1.3344428069513163, 0.5302654084829203, 2.1080870286253033, 0.06251216939171997, -0.46138457905169217, -1.4476446987755465], [1.3238383700638794, 2.5695115129905117, -0.820934937223875, -0.6470876402241984, 0.5961887986723416, -0.8303454354133867, -0.27055836699218927, -0.34983769944206944], [0.19195888729205715, 1.09481844340068, 0.022067589713919693, 0.9189100957801388, -0.41988889600049356, 0.3277985357234032, -2.138225589513189, -1.4499480667813884]], [[0.7959134126817742, -0.590149399040946, 0.5799149234726574, 0.5423442548146441, 1.3222788582368146, 0.8118590596762011, 1.0169913501666112, -0.11167133066420938], [-0.6982851765628781, -0.731558777725664, -0.4880439402887327, -1.1298291140131056, -0.5474435821203582, -0.09256905150847415, 0.2516119571058145, -0.33889079006978534], [-1.923682866353282, -0.07228295450292373, 0.22534578702721786, 1.084475594643844, 0.5778638956158266, -0.6435606861455095, -0.7237765601247179, 2.0105975223757233], [0.756637954017235, 1.831439822737841, 2.129444531219653, -0.818083666605393, 0.38526930227745887, 0.4581670048370468, 0.5596122240618859, 0.5419022252820672], [0.20205463691158088, 0.17412924424976314, -1.5024999410944402, -0.1654011831842485, -0.7476197010175331, 0.12608730560642875, -0.46754626006357575, 0.6185207511629426]]], "dtype": "float64"}, {"__nd__": [[0.4095378548399716, 0.15434618250426557, 0.15808407215794307, 0.04647334247498324, -0.22389894648110234, -0.08225063211638543, -0.24782303165357247, 0.1939935670728962], [0.007056705920986266, 0.29064559930442513, -0.6643241557990501, 0.44388946420243575, -0.38134811842537336, -0.3671410931351332, -0.09872149243086902, -0.28169229386763284], [0.14556259774914526, -0.2870823882325204, -0.5347387752594628, -0.42290140418950667, 0.6559916626337562, 0.022129164318637172, -0.5837651424723049, 0.004188701588710382], [-0.7779732073527045, 0.8947832789861541, -0.7622390801454889, 0.23937635232278426, 0.2718128321571136, -0.7250038769202647, 0.15013825920914423, 0.4985860200828798], [0.23378764288092763, 0.130508137041676, 0.4745483407305024, 0.08045227444433116, 0.16821264403852804, -0.06335577324196584, 0.3159028630077379, -0.47069207361868304], [0.39588162943711164, -0.2526507668942571, -0.545326686530423, 0.18262447788598465, 0.8464777757618218, 0.4808391057953683, -0.25782491380133343, 0.3490298359993095], [-0.2271045598606845, -0.062009728068157195, 0.04476424660853829, -1.1586091084503125, 0.09580738308124168, -0.514804594409003, -0.3486993067276218, -0.7371952053603592], [-0.7582791090731018, -0.47163220036072073, 0.41279864764929086, 0.8330206737317877, -0.012602551759759743, 0.545895466781656, -0.13199032343934097, -0.9559864123963802], [0.07495830801071512, 0.22282413157128575, -0.21430827737879127, 0.15111211710646147, 0.28627812380466033, -0.43170378381684804, -0.7384170487821513, -0.11062708267100607], [-0.10553027013519138, -0.17633160692770802, 0.49362337640068304, 0.8627943947047191, -0.2083235089919046, 0.3494433515813759, 0.4704655564634678, 0.3565782102116907], [0.5237431435648514, -0.19462198501634287, 0.36990317975665693, 1.0122409679297446, 0.40183277969002973, -0.30994683445848015, 0.30385624696727215, 0.6311901458527472], [0.18449314319481416, -0.28170034807142846, 0.7715402915107884, -0.6248835232814629, 0.2517125233279633, -0.008087787233564018, -0.5968328625887511, 0.6097248417914743], [0.17394379685067285, -0.5833990885495345, 0.3008641782809216, -0.21580021445567, -0.9513942510835631, -0.34445256269997615, 0.13177572375080784, 0.31785144313806174], [0.08278245256413626, 0.022582505633995217, 0.23499500357163117, -0.105488050982607, 0.588477301971147, 0.07432082410671646, 0.14696077623686257, 0.270399495772873], [-0.5285844201522389, -0.35461343817487967, 0.7642895570199078, 0.16765434518048983, -0.14130803930758926, 0.17074176440574795, -0.24081111753077508, 0.13928314431614733], [-0.33293552749502897, 0.13800940866958067, -0.7888512761595047, 0.665459896832841, -0.25647893634849417, -0.7906205984975934, -0.1119208472935266, -0.18549809735910516], [0.07619920930022578, -0.572625304424061, 0.19873279989512915, 1.8117838441840024, -0.6393338603716451, 0.16885778784964275, -0.15535942682521242, 0.09630414740720687], [-0.9059045189186316, -0.5755046939891727, 0.2298297676013376, -0.019209009697611298, 0.8176190902954908, -0.3523986961356833, 0.08724976357819733, 1.456560726268914], [-0.382780718208966, -0.45191674673797866, -0.017565042447544067, -0.02228133941782947, 0.42208364006126736, 0.06478793639879239, -0.37836965612431767, 0.10571077600378827], [1.3192983762893522, 0.6358868370205113, -1.3910845588116116, -0.07100845679715226, -0.43754526097548285, 0.31113125720257545, -0.08663174031205013, 0.966347873997597], [0.46561778180945335, 0.4452014462296368, 0.09328553001327593, -0.020148826653763843, 0.17008996443726382, 0.6476398957093875, 0.2671948849363985, -0.17556628291883136], [0.6375423292344483, 0.09605518452835358, -0.0388075317358118, -0.39568841933101606, -0.28299613823307523, -0.318023389863482, -1.396178579876945, 0.5084231001990817], [0.28323878471390546, -0.10713529578954897, 0.5229918692046015, 0.21899388976669598, 0.18578373948480037, -1.2373817747893319, -0.11650795980530942, 0.26609456288362654], [0.8048434158941186, 0.9201474408087482, 0.7129191228601367, -0.5808047592747508, -1.098835683543601, 0.2852347873534661, 0.013807917675321538, 0.48127662717622327]], "dtype": "float64"}, {"__nd__": [-0.012588436360404338, 0.02197549635234998, -0.1499989013056637, 0.04480096551589745, -0.044489845287418686, 0.0349985357704251, 0.04594059935421338, -0.011722214488089591, 0.031680112211393316, -0.001360703053330213, 0.07591995011285091, 0.05017429484459656, -0.0547216369037207, -0.05879018811026015, 0.17437387779906113, -0.010159549531612938, 0.09918077625655024, 0.0393385730283742, -0.05641124904961297, -0.07489737969543596, 0.01201256894943382, 0.02566544207133919, 0.061436371946005465, -0.12669334275073837], "dtype": "float64"}, {"__nd__": [[-0.29734164121305295, -0.0474331017071277, 0.8003747300737978, -1.1800892853130585, 0.21955582405396187, -0.2700229293067115, 0.21568867142158268, -0.24598945314009932], [0.18227553178469227, 0.7823190203432917, 0.17709825172854615, 0.21669997123890616, 0.18642894421744227, -0.3751567602515838, 0.2715089259304609, -0.10417328136549035], [-0.4058568220792881, -0.019273963105813165, -0.11364423672396509, 0.37673989766212285, -0.021419612498761988, -0.26472102874710196, 0.72798578786082, 0.4941938813399621], [0.6348647189671607, 0.3038336810072966, -0.0548494972173277, -0.2095467019491575, -0.1396699975260814, 0.4738453049061605, 0.05595164012237078, -0.7414215721839413], [-0.01954139885851158, -0.7080521031006015, 0.08293291703117722, -0.6019098171057788, -0.6319680641071282, 0.059289726960846875, -0.03910716290435702, -0.8308808498948044], [0.3741118590356789, 1.081548344242127, -0.7406878721455318, 0.9099248593889515, -0.3438947121141036, -0.2470453095823875, -0.08944361046887112, -0.7226386103326721], [0.6495636072191316, 0.24634681421866758, -0.37572710090658434, -0.13853857731084862, -0.7581797189830551, -0.17387887009478548, 0.4681065757199887, -0.8576448860715203], [-0.21812016447560628, 0.07339586782390589, 0.524119672923959, 0.6746652763702867, -0.22862647791833965, -0.3701534382879208, 0.22519983055629444, -0.18162739596083044]], "dtype": "float64"}, {"__nd__": [-0.03100193733934846, 0.10801836789430119, 0.16897378703352742, 0.026586757113430944, -0.019221477473105455, 0.05712619649762886, -0.18875158294293481, -0.0018249063351038427], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": [[[-1.9768286385772003, -2.310191032847325, 1.11054346701492, -0.8882027113896965, 0.545577216314285, -2.345020203372943, -1.625667201462869, -0.341904286057436], [2.7332276497221164, 0.48163877875886635, -0.7510527139093951, 1.421804442008157, 1.3624380741927578, -0.7430701705031125, 1.1460148287236822, -0.9193124422762033], [-2.850769158822517, -1.9275185562323274, 2.706275299355397, -1.5749113240064059, 0.5134616109329786, -0.8401903029796344, -0.9822430002116403, 1.0646119213295886], [4.358261241481342, 1.0995480191872784, -1.2932951930452619, 2.0466689186067537, 2.303691810198905, -0.4397689962795691, 2.369321702964852, -0.9072693714054917], [1.712347953231807, -0.8612656917284666, -0.4498004665429346, 1.1125005567548525, 1.140244809033852, -2.563953193209681, 0.20395747751453847, -1.8226652147015616]], [[-1.2404396791040202, 0.14040083783557986, 0.5160185272138096, -0.31792048999742334, -0.5710494550699375, 0.9406105104137095, -0.6590628741158934, 0.6905985414450001], [0.2560014342580876, 0.8924331766930648, 0.3531787801503541, 0.8098437393597153, 0.25365884096848285, 1.478772633587374, 1.192221597711745, 0.4290340440013613], [0.45259632611932044, 0.14910328497775996, 0.5758730947429692, -0.7659439912415433, -0.5279115311860273, 0.04956304322207668, -1.112605382639519, -0.44223122294161576], [-0.7982631807183315, 0.14537367496935047, 1.7742080343973847, -2.1914161474246847, -2.4770352809104947, -1.2378013896814115, -2.992895962906963, 0.22826211766750967], [0.9903352304656525, 2.2347357702266035, 1.0212117176794095, -0.8664581162403183, -3.898461778203386, -0.12905278432013617, -2.0576125800905793, -0.7663253997826476]]], "dtype": "float64"}, {"__nd__": [[[0.0757982574571367, 0.17088760648337564, 0.022185569934221626, 0.282513682393963, 0.448614883731303], [0.2127180892943508, 0.205553976055257, 0.35532527120641966, 0.09381095464567321, 0.13259170879829935], [0.060078660729895164, 0.4121282699926148, 0.038160066148292544, 0.35112495255827747, 0.13850805057092], [0.29475801313953853, 0.17219666894025776, 0.5035326794416373, 0.008005786963379792, 0.021506851515186675], [0.2242325135510564, 0.271941918483835, 0.13694029990348505, 0.09278540784207812, 0.27409986021954547]], [[0.12339106690796442, 0.30498787284214113, 0.08694730492443818, 0.06861909302427006, 0.4160546623011862], [0.19861999059718413, 0.1652983281806552, 0.336550643872216, 0.17800011744912533, 0.12153091990081932], [0.19020269427402223, 0.07147269039372467, 0.10620249246685612, 0.4517330886429825, 0.18038903422241448], [0.07056098630131413, 0.4202546831595873, 0.018618558714292962, 0.2684224690325086, 0.222143302792297], [0.2753006367979889, 0.02353848205192612, 0.22688660824090498, 0.45045038790895975, 0.023823885000220306]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-1.0605768454545814, 2.389427838640195, 0.7201999274777301, 0.14136984774843156, -0.19436462925611991, -2.5149560405578164], [0.7128045242850151, 1.629368948143988, -0.7288198970816447, 0.6834045108873033, 0.7225431565597581, -1.5481276661378405], [0.9357403471823039, 1.3547810757074732, 0.2284801183115689, -0.0631819947053648, 0.7879634903939723, -1.540753975374614], [0.0932807028593647, 1.0543419572150396, -1.2424353315034766, 0.9925712091532602, -0.4182296589367618, -2.080826953612437]], [[-0.10574795422979402, -0.9718966095973122, 0.31925964053282635, -1.1721411508347122, 1.0502759086725666, -0.738830321868627], [0.3563574951593681, -0.5939799434324009, -0.32290531215810686, 1.191760044649294, 0.21527844622988399, -0.3950320911956994], [-1.2515080145573771, 1.1599512739010769, 0.73281116928691, -0.3265355739495917, 0.2433110221744552, -1.0295775519649915], [-1.0148448852027037, -0.8261877084779022, 0.3534272179749535, 0.2958458940984602, -0.7453029308727064, 0.7377745962954552]]], "dtype": "float64"}, {"__nd__": [[0.39234677045088995, 0.08645408214743924, 0.5191942838776593, -0.14368122530384506, -0.056510567640953656, 0.23642933746331735], [-0.01702200463877049, 0.39694959922556744, -0.6263340665969442, 0.7128825298387483, -0.3906357431268241, -0.08234892498846609], [-0.6887713495433202, 0.3001111599194651, 0.6369256865982468, 0.5355234643093914, 0.41668748613109907, -0.7785716349786467], [-0.39108402713914864, -0.23128149562008143, -0.4673671335995304, 0.5613476154793378, 0.09224215646563397, -0.012238902112588339], [0.24376786605175332, -0.4758095189141328, -0.45417536413359577, -0.4407309950164191, 0.24881047448318536, 0.17079818955075252], [0.2890462338860885, 0.5391915587647971, -0.6786504295194505, -0.28024719312029217, 0.07376904693634702, -0.6743715535890201], [-0.2660876141664854, 0.5070144514518786, 0.551322064768792, 0.5383534462068728, 0.021508843054594183, -0.10598376852154824], [0.5397623450878077, -0.18632299822154322, 0.1691194950876399, 0.6626394620582784, -0.18378540018342632, -0.3052761504661545], [-0.004685636662662502, 0.299189589401162, 0.24120450791736345, 0.07792888660531694, 0.27821006767018613, -0.7997993719613331], [-0.4296329961585364, 0.08161305721138733, -0.4643307614556843, -0.6708372211715047, -0.603909927197173, -0.45908196161258696], [-0.08923744541218084, -0.5113129079958202, 0.11744941925049611, 0.8481120713329396, -0.07381315457108248, 0.6373494553755256], [0.1442158709444558, -0.5599117496783967, -0.12820478749431125, -0.4909173707628091, 0.3839676921285803, -0.5765477070100826], [0.6612863504598311, 0.2890939712853119, -1.0111550501906952, 0.39217070730154596, 0.7765081810699, -0.3952618558073173], [0.492626106444816, -0.8991908658506238, 0.1260783058230012, -0.8306946808433944, 0.24119549482416486, -0.8939813750412993], [-1.3164351037043351, 0.1980465842924799, 1.143944884187267, -0.2978347878208414, 0.4150827733178115, 0.05535554819733516], [-0.16175236981234684, 0.30769214255200866, 0.8212026896283878, -0.6175639935944719, 0.28194847200853423, 0.17256771496678416], [-0.7121728353300659, 0.8073554336161989, 0.9746793554319118, -0.4763859943419285, 0.8689502476658695, 0.028073059069488277], [-0.08162416942221115, 0.7517244726323804, 0.11371642760572181, 0.1615879531843119, 0.7588374873181458, 0.6441099597489884]], "dtype": "float64"}, {"__nd__": [0.0893149417916258, -0.1400549883940593, -0.15723903412447418, 0.11261412366905617, -0.09267195256430033, 0.014848354558181388, -0.10740719950541691, -0.018368652872814428, -0.12786951262487853, 0.16301740519090593, -0.10189124298867401, -0.012199426763800638, -0.018819948161311773, 0.12060977932700745, -0.14198975560984484, -0.0033342032421233916, -0.020923426757877228, 0.052992554121547214], "dtype": "float64"}, {"__nd__": [[0.23600838959049747, -0.23187326697073876, 0.11979299556265305, -0.1873155911210439, 0.23405761869594652, -0.051290082912325674], [-0.32778454686387615, -0.013778511307817456, 0.18189198065476067, 1.1329413006576088, -0.2456159353929058, 0.15332903221967764], [0.7315183608080948, -1.039084338003852, -0.8715190988173499, -0.5943296750579143, -0.05711827141745718, 1.2199883907465354], [0.19568348619766415, 0.604037670840102, -0.11101967534821768, -1.2318618821495406, -1.1568191797203602, 0.5093096178364318], [0.0924297712526503, -0.6492495839063404, 0.5586603409999908, -0.7694774877737887, 0.5671210616014323, 0.725670424832987], [-0.3824116877781579, -0.5540585190327533, 0.42216352531588125, -0.5476442893153087, 0.4309432285988196, -0.5976193889818185]], "dtype": "float64"}, {"__nd__": [0.08075785602641582, 0.009386694355339925, -0.04327783766616464, -0.08956846826243019, 0.023757476985481286, -0.1892660633334253], "dtype": "float64"}, 3, {"__nd__": [[false, false, true, true], [false, false, false, false]], "dtype": "bool"}], "e": {"__tup__": [{"__nd__": [[[1.26826885126928, 0.03418487587414079, -0.13603673544930034, -3.5731693979732566, 2.5036290694486363, 0.29815999850863806], [1.0852459133905135, -0.1857550899336133, 0.870876248626313, -2.142844154041821, 2.0245756336834653, -0.3370026772892359], [0.8646232292398813, -0.4112652187319613, 1.151447520386155, -1.9459350982872328, 1.8298468344791352, -0.0558200423010036], [1.2176172705490118, -0.08515324043496261, 0.6778563085751514, -2.37416855062127, 2.166490778812003, -0.41815946462745024]], [[-0.2523149434866407, 0.5438483681955096, -2.416785273416289, 0.12183258477681742, -0.6372367723201157, 0.00723156038401318], [0.14246402653041185, 0.8221610789356695, -2.5838251681974453, -1.5253427077239137, 0.23779126545040555, 0.19545474790133008], [0.21790033525926733, 1.0796835462719354, -3.4766801982738986, -2.833414251564146, 0.6747045275530644, 0.6653022173379872], [0.06042293664635219, 0.57962002357634, -2.305375088727743, -1.2478081077835645, 0.3030604047632786, 0.28557834205076704]]], "dtype": "float64"}, {"__nd__": [[[0.6688342573045688, 0.33116574269543114, 0.0, 0.0], [0.4986875058425662, 0.5013124941574337, 0.0, 0.0], [0.5418659488796588, 0.4581340511203411, 0.0, 0.0], [0.49128354865741025, 0.5087164513425897, 0.0, 0.0]], [[0.23372661499211253, 0.2328202867939538, 0.25558328407340986, 0.2778698141405238], [0.2516265721149998, 0.2502056865630801, 0.287954352258497, 0.210213389063423], [0.3579396244734365, 0.11725870756157784, 0.42496429043711315, 0.09983737752787247], [0.1888888008814953, 0.18417784428160391, 0.3141097261217534, 0.3128236287151474]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-0.2759641722489213, -0.39615416922780433, 0.6783599277889846, -1.2538203551751494, 1.7134381385954434, -0.31158077278064505, -1.1381675222664627, 1.3860787495037976], [1.6629310372409845, -0.1877719999739882, -0.8910728393162252, 0.016052968645974074, -0.690411734841456, -0.31203353977523773, 1.5584293956790127, 0.39190098723383365], [0.14539382420037475, 0.3460121962194285, -0.05808980630250469, 0.841567862220092, 0.879290130895175, -0.36971672115570664, 0.10508839204188516, 2.413607678811649], [0.4215991461724016, -0.1858533833881879, 2.049691752795071, 0.009432553838472772, -1.0973019243764386, 0.674580859823949, -0.8998261496510739, -0.49722251447843724], [0.32632538632402314, -0.12676692484843896, -0.7634866074150394, -0.5509565875585889, -0.5933760026769729, 1.1661864544428544, -0.7210002108727575, -0.57221906070492], [2.038354369963076, -0.5537643267840531, 1.4894177085203448, 1.3157322025559548, -0.1527516209306077, -1.4193913341554476, 0.45301413108094896, 0.672675147079706]]], "dtype": "float64"}, {"__nd__": [[0.48390224086948164, 0.03594711884760754, 0.6022787426424135, 0.6951793035263545, -0.5271910204274002, 0.714744153517866, 0.08727117708038547, -0.2828720279864182], [-0.15931386404688114, 0.9387860707593739, 0.49201498880437416, -0.8945229797622173, 0.04172875453694038, -0.06992497428961113, -0.08779363597846927, -0.25657761337018276], [-0.7281507372672719, 0.43082215635291593, -0.4698448700481566, -0.18502055458800334, -0.3011926428735698, -0.36391193893695883, 0.37161833778028874, 0.191887079232164], [-0.33708081683871055, -0.47240534575398313, -0.2347072054211306, 0.48351208889085784, 0.0629606878339594, 0.1165418460262346, -0.16408183403185578, -0.12894020689994282], [0.10827186332980633, -0.9217125050585258, 0.8060290561159198, -0.39382177587890976, -0.2841571925301317, 0.6692485103577451, 0.11121686263242439, 0.02104177709402124], [-0.2916596485503575, -0.7805846981057952, 0.09247267758775639, 0.5976306943623783, -0.44982760335382127, 0.6813774308943039, -0.5635710643740016, -0.3670055628601073], [0.25294577476115887, 3.261417509192931e-05, 0.34544400055913754, -0.3708690459148312, -0.5158808102588665, -0.9335313505720727, -0.6018556248255262, -0.5380538773189822], [-0.713482810261243, 0.15423058767007236, -0.33194258271831195, -0.9941475309967285, 1.241741320974215, -0.41742683063561653, -0.6277546637100071, 0.6198941179103105], [0.5827169144128516, -0.5756704646787743, -0.7369909437470681, -0.5000333751697897, 0.5647428967337789, -0.15563038364628237, -0.40434892918226667, -0.014186206372148907], [0.6020957474512476, -0.6947751859529143, -0.19190089027658594, -0.6917378099615732, 1.086077748664925, 0.9091597671841974, 0.47558103956669284, -0.06670079228933977], [0.32956330790720667, -1.051006755500075, 0.13821338748697856, -0.6680826077149622, -0.6623864043374952, -0.35236295624104663, -0.17510837334431878, 0.28710962241196436], [0.3503393741597636, 0.13133628840711098, -0.47216829703217683, 0.33799339400683337, -0.06047349111207816, 0.5901144265797048, -0.45856532256062926, -1.381612543194262], [-0.23835460217521318, 1.0756317469892258, 0.23449367654279782, -0.09716458533714593, 0.4418181516665029, 0.2515912319327309, 0.23067065172085532, -0.2567728109586294], [0.03808551984687304, 0.10059600209627664, -0.32133088683850713, -0.5593650818101573, 0.0695113305205037, 0.48422506919795594, 0.3236521622221911, 0.24698879483318323], [0.2521574699612575, 1.2920013581470011, 0.12994186075516267, 0.5210190435323223, 0.9785499874862896, 0.02653155342335402, 0.1322650242441044, 0.5220232561364786], [0.8438316354697718, 0.15655265243576544, 0.1815089099153958, 0.04541978819330375, -0.6027995956554068, 0.5588339325723576, -0.12624937721175125, -0.8593762968800659], [-0.07206835733548929, -0.8635149302231145, -0.5023479773256286, 0.01054199211360095, -0.45846288978447214, 0.5376613616458938, -0.7320696525600812, 0.13456643917489877], [0.2821340502356503, 0.7337358521948905, 0.4649858851627077, 0.12060222641511607, 0.04433677454983984, -0.025015973639403326, 0.5737376622579899, -0.3238992390022519], [0.26511929885859825, -0.04206349043569527, 0.12519927038761974, -0.12619319025929507, 1.0726873621399449, -0.32336658224324305, 0.13190423707175772, 0.14344391042218663], [0.3096183823879239, 0.2308765925229798, 0.21801717818772476, 0.4967490405742481, -0.7466327377195098, 0.010867119924986918, -0.6108407806483465, 0.12229788905887301], [-0.5853036067059182, -0.47390886298126317, -0.024123123467763807, 0.7044228725337601, 0.29499208782230424, 0.15466084554933332, 0.7080732227769581, -0.4661303651426896], [-1.0395114300311799, -0.20535021780085122, -0.1324798988655461, 0.624018831090813, 1.0762738873761029, -0.6581669209839681, 1.2454532159783975, 0.7890316062247847], [0.4261382767268479, 0.19090401750691738, 1.2199429537719888, 0.04718745869179071, -0.22992648205914254, 0.07648925736810581, -0.677665905907928, 0.24299201880479332], [-0.04790681917704087, 0.17229955199589508, 0.18711761632304758, -0.5223954533465682, 0.6651608196027081, -0.5003354331580615, 0.45118121766094743, -0.7895378679619669]], "dtype": "float64"}, {"__nd__": [0.03858772738268019, -0.08055641521426755, 0.0057244624282100755, -0.06367143260963386, -0.2168475081769634, -0.10997687272159129, 0.008372883459774622, 0.06391448233508122, -0.14539361819853164, -0.21098472424674763, -0.08568490922447702, -0.01960392477823167, -0.06178174625916948, -0.07942455642986827, 0.08660274310384469, 0.09813162749311737, 0.07571935711188085, -0.09225445507829416, 0.009168804514012757, -0.017823215261918223, -0.14120382138249235, 0.08979007849930247, 0.010336226305510758, -0.046951435332984376], "dtype": "float64"}, {"__nd__": [[0.7164677313353808, 0.38960349250830756, 0.4356896130486864, 0.6076969732258107, 0.3665826275783786, -0.6250121755854643, -0.18682479940457702, 0.5091421057940031], [-0.37694149820517764, 0.025932467806260892, 0.2062340021104839, 0.7280824779114486, 0.6566590484844504, 0.4564143482076686, -0.08607812118376004, 0.46450347244620094], [0.27103704108007143, -0.07398950938324696, 0.6856955203602628, 0.2540516338185928, -0.37132892373411847, 0.6397070027720061, -0.12107020668376538, 0.30323091572613187], [-0.7559036413729839, -0.0035439942837559952, -0.4670954789722612, -0.48051284050523924, -0.94959598035629, 1.0772404131847442, -0.06953138019165649, 0.12444020373388494], [0.5592459027055126, -0.05575164507762094, 0.261815316471342, -0.6745603594828219, 0.10308422720601963, -0.20214448841173546, 0.2550198740156, 0.09515830391898705], [-0.6882303498511114, -0.9214177001931391, -0.5030401468023594, 0.37295389458248723, 0.21682992053884279, 0.20577601859086822, -1.0055885295690623, -1.3407652484241315], [-0.30814327373093475, -0.16278748329039514, -0.29832958402570026, 0.18051218250115453, -0.08253569327655108, -0.59012141532208, -0.43727374938311037, 0.07405963660765558], [0.03869812482381857, -0.03490786164299996, 1.135785218534696, -0.12939005078782234, 0.06189215601678033, -0.49633469758447146, -0.17801452124585088, 0.2634747187867191]], "dtype": "float64"}, {"__nd__": [0.056949809001393896, -0.01553387778553815, 0.03967377373535125, 0.0056582345403827315, -0.18801957384853676, -0.32744750252613536, 0.23111715675830524, 0.08309166106085122], "dtype": "float64"}, 4, null, true], "e": {"__tup__": [{"__nd__": [[[-1.5460145182103644, -0.8958705921351486, 2.820321870148696, 2.3701117239389746, 1.1961352738457371, -2.5951139753884096, -1.6140035152734156, 1.8699089055794764], [-0.8684227943500487, -0.4916987563805677, 1.7545361673854827, 1.5793866998582142, 0.808929536526633, -2.9875029174168226, -1.2324310078438612, 1.435521255889745], [-1.3332003551192655, 0.34787458505147667, 2.109883118870733, 2.587868259569906, -0.5742097574967494, 1.1893742631262478, -0.4387802719582383, 0.8138075866675596], [-1.86071866578207, -0.53697042637495, 1.899933148443247, 1.770467075755286, -0.16466716705039244, 1.4184758231779688, -1.0832691687610654, 0.33748848296966827], [-0.28452130441294393, -0.9383011161441879, 0.32925190434358725, 0.0640115348844761, -0.02278056615169652, -0.25169261203569965, 0.02128370214908179, 0.04361991490086469], [-0.3118404778702259, -1.0170627225307856, 0.5245929737693364, -0.32509857945613385, 0.039259765098185156, 0.22813803873102417, -0.14635796135010787, 1.1807330763884922]]], "dtype": "float64"}, {"__nd__": [[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.6651291332712284, 0.33487086672877164, 0.0, 0.0, 0.0, 0.0], [0.174874176926733, 0.47621174385422627, 0.3489140792190408, 0.0, 0.0, 0.0], [0.38412992299668797, 0.07914731171449299, 0.5094452926770314, 0.027277472611787708, 0.0, 0.0], [0.2657550508663803, 0.18289518417874853, 0.24136489676917608, 0.10302240686772132, 0.20696246131797374, 0.0], [0.12841618142505057, 0.0645769799256405, 0.5072249113440288, 0.0823379546496176, 0.07522028333029174, 0.14222368932537074]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-0.056763679670587215, -1.7380382561234844, -1.275476926728536, 1.355910475560656], [-2.788177396734619, -0.3243311751827612, 0.8685643707938588, 0.14323113956807265], [0.7547927960126413, -0.1312493323652416, 2.3934515844200646, -1.0279959857034178], [-1.3622270260269171, 0.2745323457384551, 1.4295806440689964, -0.055891882049737746], [1.049004893695953, -0.5308372772821607, -1.185274456798664, 1.3999205889632227]], [[1.5687130317863218, 1.5331289389866234, -0.438252549382261, -1.1552072496527401], [-1.169372932977615, -0.2246351948398268, 0.5824377910705448, 0.5091717251007895], [-1.4344057124813594, 0.19517282113137333, 0.017730241573760675, 0.0901277427551174], [-0.023697257578161837, -0.20192024059231037, 0.5932751351544383, 0.12685956100129553], [0.22541081625333215, -0.6907558179163049, 0.39577778077264153, -0.12115202407949936]], [[-2.115161206719354, 0.48361008112032944, -0.2643015661031351, 0.9262818985828198], [-0.1620987660364429, -0.09574252299710637, 0.2116899379073339, -0.011856407610606527], [-0.44692981301642565, -1.7932005266838178, 1.8426882782588792, -1.5787861979761342], [-0.3415219400245493, 0.6531383002851368, 0.3800170124042527, 0.5672684708544655], [0.7691897816929107, 1.2633727016452962, 0.29728336176131304, 2.9066426225656414]]], "dtype": "float64"}, {"__nd__": [[-0.5067058980448064, 0.11368945200414, -0.18042646117799233, -0.27256865376232475], [0.14363258488838745, 0.7291203356853473, -0.3551042945244026, -0.3554778590362922], [0.3622444194832113, 0.27199704642077954, 0.978878453126227, 0.5139399834219576], [0.11479564577199096, -0.164686517889315, 0.10763140082807632, -0.08760851207843935], [-0.6825454925346083, 0.004117382859142945, -0.0875610922818704, -0.11627039432475283], [-0.21465438453609048, 0.8450347102092162, 0.34813309134478804, 0.4684313734717822], [-0.30383561443708446, -0.5113107588921103, -0.1414418258969314, 0.1220358906848496], [-0.17266446481637676, -0.09411397806181436, 0.48070976482654115, 0.1496120581179875], [-0.49200746611952306, -0.21399964146890757, -0.3287699903217287, 0.09046919442183274], [-0.5836258816655442, -0.08937367969887007, 0.34632087787140425, 0.014361981362658429], [-0.45693823430769864, -0.30820702815369233, 0.062175497761723715, 0.24154077368682697], [0.418535169042657, -0.07260571314071264, -0.6586248999228429, 0.7124963075346774]], "dtype": "float64"}, {"__nd__": [0.02978863050101901, 0.0406071879762164, 0.030551662807689645, -0.10374903284548731, -0.03193045329653708, 0.025725854377995006, 0.10326280972960145, 0.006018184941311414, -0.31199484617298706, -0.024135127104889735, 0.03526540938558664, 0.129388912877986], "dtype": "float64"}, {"__nd__": [[0.355658125359589, 1.2451043829213815, 0.7121711454360035, 0.6738319038069105], [0.4713936735998225, -0.6912176442755741, 0.4235252462415172, 0.29974205849822805], [-0.26301027282524014, 0.2933229860409456, 0.20903307322390716, -0.23250979468601027], [-0.06169564237958817, -0.13164740984142426, 0.11523954598959446, -0.2122876933282573]], "dtype": "float64"}, {"__nd__": [0.05449131725222098, -0.1917587857289499, -0.02819282236523998, -0.16846874347825347], "dtype": "float64"}, 1, {"__nd__": [[false, false, false, false, false], [false, false, false, true, true], [false, true, true, true, true]], "dtype": "bool"}, true], "e": {"__tup__": [{"__nd__": [[[1.9324623198445374, 1.2522205159189994, -0.5667086813231665, -0.5086226853565381], [2.3882443123534762, 0.1473887497691202, 0.21034151455330743, -0.24594836969962702], [1.795122486961191, 0.42687492670097094, -0.06610578784014112, -0.312951996546103], [1.7486495064192062, -0.177713996676228, 0.30302576085953653, -0.17489193366567996], [0.5245944715747708, -0.11051874993359617, 0.03058856879837778, -0.209548008001113]], [[-2.9293647677567383, -0.5545215715289592, -0.37031939618513027, -0.11794390474873931], [-0.2422942228905782, -0.5206750545835488, 0.11675208822866298, -0.1228973192916779], [0.1088686458728732, -0.4906077635548466, 0.15968830225116867, -0.12470441291795678], [0.3253753263412249, -0.4889410403590683, 0.19979003959615474, -0.12505556985601549], [0.022462082891174538, -0.49517697948688116, 0.14683792165177692, -0.1243877081437166]], [[2.4772397666268047, -0.10738741112088557, 0.29344142289612896, -0.24790343970004972], [2.4772397666268047, -0.10738741112088557, 0.29344142289612896, -0.24790343970004972], [2.4772397666268047, -0.10738741112088557, 0.29344142289612896, -0.24790343970004972], [2.4772397666268047, -0.10738741112088557, 0.29344142289612896, -0.24790343970004972], [2.4772397666268047, -0.10738741112088546, 0.29344142289612896, -0.24790343970004972]]], "dtype": "float64"}, {"__nd__": [[[1.0, 0.0, 0.0, 0.0, 0.0], [0.4950729781762171, 0.5049270218237828, 0.0, 0.0, 0.0], [0.6658238220815074, 0.23414927857564818, 0.10002689934284438, 0.0, 0.0], [0.4138888909831309, 0.33012177177055885, 0.0968774012310085, 0.15911193601530188, 0.0], [0.2296057381038719, 0.08539517060245069, 0.2576121850411178, 0.1274926108704187, 0.2998942953821409]], [[1.0, 0.0, 0.0, 0.0, 0.0], [0.39893796010496313, 0.6010620398950369, 0.0, 0.0, 0.0], [0.3045765191187834, 0.3174087233624775, 0.3780147575187392, 0.0, 0.0], [0.2568006476487113, 0.38081593127956054, 0.36238342107172816, 0.0, 0.0], [0.32605064844944986, 0.3472450473343513, 0.32670430421619867, 0.0, 0.0]], [[1.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[0.03419276725318417, 1.3597475403099617, 1.2247210785859324, -0.5103070767876675], [-0.2979695111064471, -0.5273841930334252, 0.5697263575719601, -0.056064439045617594], [0.7468856162565439, -1.8473247989741095, 1.5665487746995206, -0.09643216015562055]]], "dtype": "float64"}, {"__nd__": [[0.34018922663707307, -0.06828316698841387, -0.18954928353742664, 0.23155507929879338], [0.4122567637650565, -0.10126493534672576, -0.07639308928509854, 0.342849305404629], [-0.4351703209735856, -0.7571917518656978, 0.197490931374765, -0.3352829118439397], [-0.9601702950590143, -0.40702683197267975, -0.2337987794463735, -0.5966012387161306], [-0.7462319420315169, 0.018318913472402543, 0.4486246283638738, -0.11656603898022842], [-0.3717980147544224, 0.19249690437395414, 0.3586179035971919, -0.1500052992442387], [0.27233390396044643, 0.5214377382914769, -0.10347821810416198, -0.40675777099078614], [0.17382529925775475, 0.12377287048142377, 0.5494063842072042, -0.6422903894026725], [-0.33080646517777385, -0.41908348035783727, -0.8670074231164258, 0.0632172775984981], [0.263902106247762, -0.36939501573790323, 0.6928235372480793, 0.41096216833021765], [0.31368823941776763, 0.20085354572048494, 0.4778347822243175, -0.6659899197715511], [0.30696482912493217, 0.30138841676672395, -0.8838592885714874, 0.17351505102718986]], "dtype": "float64"}, {"__nd__": [-0.025042134670996842, 0.07815226960616994, -0.04390621876376686, -0.0018240857649100333, 0.03428515173176555, -0.08762616887442078, 0.05985966481803844, -0.010496318852366824, 0.04924826236792429, -0.052183750633678784, 0.10862015432775177, 0.060520197842947425], "dtype": "float64"}, {"__nd__": [[-0.08901251235966837, 0.31597857854680506, 0.6298775806793125, 0.8955877567489944], [-0.7867881852201097, 0.44156590581125976, 0.23253425425669064, -0.04693039009317199], [-0.5033324674885357, 0.6285943067365718, -0.6308689967222852, 0.28347273286737446], [0.6509339981013448, -0.7998346440257398, -0.15125892024163118, -0.6546084087581496]], "dtype": "float64"}, {"__nd__": [0.024405410803590058, 0.15143751306746547, 0.20235560291721977, -0.17781144428835152], "dtype": "float64"}, 4], "e": {"__tup__": [{"__nd__": [[[-0.09993036035841012, 1.391576219090473, 0.46229377572452024, -0.8619697426697096], [0.020746038342752696, 1.4342451290630842, 0.2597968530918138, -0.8425180394771148], [0.11609577822302465, 1.4367379616467126, 0.08525852304618375, -0.8000618932457344]]], "dtype": "float64"}, {"__nd__": [[[0.25027119357266236, 0.3622604664713723, 0.3874683399559653], [0.3951099375234365, 0.2982265079036315, 0.30666355457293204], [0.4504528647335634, 0.2961042436210046, 0.25344289164543204]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-0.4828994055152588, 0.5169365582122687, 0.02161206220729548, -0.11063522406501289, 0.0687923704514405, 1.5695009171706567, -1.1062519664390633, 0.887733163892781, -1.5125967976357788, -0.4674409530656326, 0.1635974663667906, -0.8516802796258283], [0.2051373748588469, 0.7286924130231177, -1.1937425853510049, -0.07737187402361104, -0.37656624380573456, -1.940383750108247, -0.1870741593687321, -0.7977706641384916, 1.3390848762278962, -0.8755627219446027, 1.6994721983325756, -0.5334468091161984], [1.9381440416467546, 0.5393889643174486, 0.45384663864571767, 0.8057140372434636, 0.9023076736227501, 1.4032914160112737, 1.7327082739402266, -0.34671371601491774, -1.250134840661803, -2.671756314672663, -0.3872530433388884, 0.7602483316290657], [0.3783622610501528, 0.11518598504297725, -0.3501964891053096, -0.6399925073998515, -0.2836743665965275, 0.38897601384866326, -1.7646073410001109, 0.7289218423079096, -1.4584585195145676, -0.1582504219456442, -0.6345183311758219, -0.3695299930364056], [-1.047759199579851, -1.1557211984076157, -0.23337388951821267, -1.088032491064515, 2.4552684940445824, -0.4210308622095371, -0.6616194265509461, -0.5554501579040428, -1.0777614685478931, -2.462776380293462, -0.26251774179364884, 0.3036781808383792], [-1.5107386734307533, -0.0015084293193091856, -1.216866519157122, -0.04295723664760023, 0.3519157300894203, 0.47802792648450976, 0.4276074663440487, -0.7118833694527169, 0.8723000385221371, -0.45599626107499447, 0.3419435745509745, 0.6964702375276849], [-0.31238669103838407, -1.2928166248758384, 0.6050906711403551, -1.125698335088863, -0.899648652284072, -1.1112617052061289, 0.8338878305671076, -0.1720209649202813, -2.2575388980458806, 0.7259496864877375, 2.0618931968003893, 1.7909938086046109]], [[-0.5771153198389466, 0.0680386477378159, 0.49938811388986465, 0.8838751739940897, 0.025115592684801253, -1.1043341175393315, 0.7447573420627167, 0.6051027468781527, -1.3408242997700495, -0.37759608861586347, 0.2584249738264761, -0.013537040158141394], [0.5134997651079655, 0.7845666975151093, 0.5305721846024586, -0.1261440721745425, -1.3604287739217926, 0.6042048753294543, 1.3895822171754864, 1.9639082719526526, -0.045246283116937853, -0.04780886952570091, 1.434280343550896, 1.4265428153589603], [0.32809247121433227, 0.700002626465831, -0.7743433618140189, 0.5709987234848108, -0.8394697920019472, -0.15864133262379854, -0.13671644398724112, -0.482520095630155, -2.125680935482724, -0.8402491540892708, 0.31922249107749323, -0.07493984446773366], [-0.4077997729778312, 0.4145791813491706, 0.6745865076768218, 0.8333793506260436, -0.821886467639888, 0.00623406170298788, -0.3497337522178352, -0.8942205459170451, -1.5127911505986935, -0.11460391500807023, -0.889385046649373, 2.136916506597393], [-0.1152332459411415, -0.24113189194574752, -0.4143143399457627, 1.8589039887604408, -0.9585285259673488, -0.1776804116011882, -0.7863461058975564, -0.14136204354355847, -0.05249828991261809, 1.0822009059673012, -1.4329453261908138, -0.037632824117134145], [1.5996841350692554, -0.5265783549171041, -0.19427301748741202, -0.9037776239918246, -2.6351055336003513, -0.1214481608057499, 1.1588192785966591, -0.48466804587942536, -1.863440071819456, 0.2495400635915037, 0.06212856579081349, 0.9835210036072138], [0.8935942839550007, -0.2871144194711792, 0.38293658147100573, -1.8163076030037426, 1.096862391849496, 0.16897082232891666, 1.0682864034105377, 1.0457680339736728, -0.30434552864159387, -0.16643412157405474, -0.8917707127170484, -2.0833543861034]]], "dtype": "float64"}, {"__nd__": [[-0.044691576924778986, 0.04099354073509142, -0.3308420348280358, 0.20844192830018088, 0.024440941876955228, 0.1309851557262733, -0.23599090368248854, -0.534454093664314, 0.22827610516332142, 0.1782015657149655, -0.0380420869900401, -0.13143193359595712], [-0.08820572748393311, -0.046181472632026314, -0.1577468848098516, -0.6083889090767981, 0.23018045652695415, -0.4535270133911178, -0.10494194373834784, -0.09708334119869315, 0.07331545852427618, -0.48902190098323417, 0.3572427256512696, 0.19195506166025664], [0.226411172413898, 0.37580194420624063, -0.24367827040448226, -0.9445827692165166, 0.09607701426190694, 0.3902402962571227, 0.34548172525499515, -0.16364439580576776, -0.03021880799021394, 0.08318888369013426, -0.5446335796983643, -0.20233496026984085], [-0.19297370344080106, 0.4731599195024352, -0.19715573281670784, 0.4697252044506833, 0.002718492958070056, -0.5041482409224389, 0.6643424583663211, -0.29232153217366175, 0.1850202051969937, 0.5116760961043291, -0.160544065702418, -0.3691726386928383], [0.3122762159859364, 0.019625196614924328, -0.36819982396467604, -0.3243604953943941, -0.11689840892172168, 0.28920590727038364, -0.0861174094207428, 0.19635601964283686, 0.4958141625971498, 0.38082751478019927, 0.43127360735059495, 0.3695717140675768], [-0.48255811749565997, 0.36688305654771586, 0.09856216582143557, -0.03731935916351122, -0.3253947300240556, -0.47670230790454193, -0.3333299589112598, -0.006409198448181384, -0.35252637231226797, -0.24494244442108357, -0.18501257415745095, 0.19996159355328252], [-0.08855394170367716, -0.04376828132271478, 0.39786893429329756, -0.09637289216169907, 0.17977268014892395, -0.3171892999760916, 0.09874390268967576, 0.03585443436894509, 0.052602712583575205, -0.13991260677879186, -0.3091795175768239, -0.08985876722833874], [-0.1403922281916745, 0.3100126037287868, 0.27235031538850424, -0.36063636244225433, 0.1881969232639915, -0.14391946679138684, 0.13767610338665096, -0.14446724585891735, -0.4415302527264514, 0.30488024787251805, 0.13568861508381067, -0.3677061331716583], [-0.567423627618648, -0.16026715222472573, -0.3545530860025426, -0.10917439120514226, 0.18608301149854103, 0.4407254754378022, -0.10704047169340426, 0.4132059180932403, 0.34930329346418154, -0.13032591502199695, -0.5001069474807843, 0.25607721544290274], [0.1831221999203622, 0.7064763827242958, 0.4295325931552375, 0.23530701893919664, 0.5072167922322641, -0.31358140082187574, 0.05909977377827085, -0.08086584287685583, 0.5240067455917666, 0.17094417030959358, -0.9163229357627388, 0.4128765183788636], [-0.2700901591805069, 0.23545046948395768, -0.14111831350689213, 0.07626611769184055, 0.19376076998001088, 0.06161936980817626, -0.28260010278214626, 0.42359371133816454, 0.12312549094369009, -0.0008474008749699727, 0.21619832743495201, -0.1561038332624396], [0.5704759989463858, 0.07230258115797497, 0.11464718960095781, 0.109231785583863, -0.26862490213096696, -0.34702420044792215, 0.07054133554563735, 0.11120203880321143, 0.40937488433942837, 0.14101795054055186, -0.3210919521176887, 0.16362978807476888], [0.2539912995563626, 0.3067134690408611, 0.10590385752810345, -0.36068555978958594, 0.35421083336001313, -0.2137935472498974, 0.17791587010192053, -0.46612699849725564, 0.23517885414944062, -0.22975003888928933, 0.33046725697939444, 0.534933715314377], [0.35653403832840674, -0.3448231709948685, -0.277746495519032, -0.25578182292595464, 0.20413141257652828, 0.1089624777179919, 0.05276500052045223, 0.0979839571097598, 0.17903995009768597, 0.20046160928993234, 0.5118698486215664, -0.0119868032629266], [0.065758833111143, -0.552265105096864, -0.39280360350685545, -0.20388267249399541, -0.3519402398849059, 0.06994811640109562, 0.3158870085764627, -0.1273899304131927, -0.14262977228459087, -0.15073693440011735, 0.04182058189501945, -0.16111326376521748], [0.13018237303668476, 0.09226698849210431, 0.10510894419049467, 0.2308223076555442, 0.010825897979326832, 0.04434755258056512, 0.5289008574852074, -0.01285489852635082, 0.24010662706283742, 0.016710697834313247, -0.13715538207916297, -0.13704261883302765], [-0.24612015246450936, 0.0034052053812987674, -0.18144165904786208, -0.12721477151210564, 0.3655065108762535, -0.04040340022843048, 0.3416836171882035, 0.07925299551164308, 0.2581394294625833, -0.18376455432324768, 0.04360490055260507, 0.7900307277175054], [0.24748267568727084, 0.3001749367247272, 0.06901866036519716, 0.25616221233307424, -0.34986233472910955, -0.2566372402727353, 0.12718970531985643, 0.24445338533917532, -0.12472710113050019, -0.058101629929278434, -0.11053353836410136, -0.05851650356826969], [-0.3625060387769176, -0.43242270449221837, -0.027332854480561802, -0.5243357112434713, -0.296821906735335, -0.20157154923559695, -0.3196364353508795, 0.06737785853804155, -0.13823529529370818, -0.029022605236358145, -0.308195497391594, -0.22810928370831784], [0.14787334492192006, 0.10278264200630079, -0.1460059638748711, 0.10237798289057877, 0.04922708635556634, 0.04047822858284966, 0.47080745886273195, -0.12812445056339064, 0.11770010380466153, 0.11808396130064083, -0.07880921401854547, -0.35827853337521326], [0.19818112076206887, 0.06919266244829614, 0.005218993286629959, -0.003006882470604538, -0.003930964618030741, -0.278870858396258, 0.07661021002286014, -0.5523067551317825, -0.01963905894908801, -0.46108247819789844, -0.11821383467373395, 0.20162898302570337], [-0.0910960113932053, -0.15754858598628654, 0.1400946347614404, -0.09508112873348336, -0.13694498574879813, -0.13282069110331898, 0.43848297967808697, 0.18396271896680091, 0.1286931569884151, -0.3728366820647699, 0.16969252135744556, -0.4041913188414551], [0.12725712230708652, -0.35154664029551574, 0.5686811151686925, -0.5223842407232268, 0.34907138680967925, 0.22268585267276322, -0.550325765956302, -0.09009641546462512, 0.4152660611067791, 0.3226820897974793, -0.5156924937908415, -0.17428527709768865], [-0.37044470502351984, -0.3258501621667936, -0.21118897878515902, 0.5336627159490973, 0.021206260855075917, -0.08662886118076522, -0.11936925364269539, 0.1807822876582676, 0.18819698929920733, -0.2948703696264469, 0.4209598558549487, -0.34404946339177467], [-0.1024057441428288, 0.11581422825587877, 0.49378452933393585, -0.12324069221228559, 0.06633695026391692, 0.37125016531481453, 0.16749473408101134, 0.3186263246092891, 0.2919468652993174, 0.05844864265011942, 0.16970786628502485, 0.2554439029644827], [-0.02785469622615678, 0.45991914759942953, -0.6152978248368899, -0.295686012385097, 0.41360383138837675, 0.10859689615346924, -0.21265776801379352, 0.09396919046796733, 0.30655774729937807, -0.187051729492656, 0.21100307223597278, -0.05595973010172917], [-0.05331177103197464, -0.040024680358772144, 0.31419798155900536, -0.35505548650001817, -0.12940597484396282, 0.1795858991513065, -0.3847959497876344, -0.4933249794406737, 0.15158734088880588, -0.01860361589369739, -0.0631942561866231, 0.023660368598702144], [-0.31064711929898847, 0.8547525484046947, -0.3142180629733444, -0.14959067912867338, -0.445449232037638, -0.1437502419723626, 0.4388538136631845, -0.013324291742213645, 0.1594366171729463, -0.16401997572522112, 0.5249083093714864, 0.030161902257677398], [-0.3934866123177206, 0.21199233955758853, -0.12988192541949967, 0.1630839906727394, 0.02274381324395044, 0.06003237215298819, 0.02196037744758996, -0.2906589556602946, -0.4696759229211616, 0.27367393088818526, -0.05973580585973793, 0.0665784223574692], [-0.1582833167971774, -0.6260966523804822, -0.4022407848933063, 0.3993904185037447, -0.25482417699389875, -0.13408308110103948, -0.4595087333855166, -0.17769710412489673, -0.13124207560789936, -0.0860402978018006, -0.04175912222711572, -0.20917995156437102], [-0.1921315929823702, -0.02404134478460217, 0.07683524760903827, -0.06269274634120299, 0.5026359681705075, 0.04141131266375563, 0.02806441351501827, -0.21715192127213503, 0.28767178829648865, -0.32759992332183846, -0.5060418733577504, -0.31327350481229915], [0.16492756634522182, -0.36419711708834596, 0.3672182466286708, -0.027801173438435425, -0.4758939098813353, -0.08983945470212597, 0.5138572819963926, -0.2792314536792183, -0.4141995563842073, 0.13239775624194333, 0.37052007709822093, 0.28349958124084834], [-0.11997828394725335, 0.034839581714439884, 0.07604936576727006, 0.3307451108799605, 0.6231256989467807, -0.05380184627698594, 0.3683046577313157, -0.12466882591872853, 0.1552197471534603, -0.30144058130691376, -0.25069039142733496, 0.10307532842044291], [-0.2905131378200561, 0.24462199688814937, -0.18667801042421175, 0.001751667621827468, -0.04655025173365054, -0.12151048706186772, 0.01779551901820169, 0.3022812813989077, 0.9100756446344722, -0.37754231888909934, 0.06855330836639587, -0.07414418404930548], [-0.29546474739545103, 0.06442102366665522, -0.28508250305475297, -0.04492278089401323, 0.35044354425948504, -0.6277575347270642, 0.05282138312059143, 0.11254761860794563, -0.30868105174622446, 0.13569345220645918, 0.08399780912412852, -0.12615181450227242], [0.3421387706746478, -0.41367389157979206, 0.17146273422087413, -0.09153895336119, -0.3383858278194417, -0.07474182936510601, -0.1262584769302013, 0.2241629924037185, -0.47004925526914404, -0.07988483363531665, -0.15819922995897195, 0.45033196565632344]], "dtype": "float64"}, {"__nd__": [-0.11899964583818867, 0.09865162752936629, -0.18289299262502545, 0.027081647431505863, -0.02099590085605096, 0.03031649414459746, -0.07782979478466678, -0.20270565257994924, -0.16222025092709766, 0.12343794906402028, 0.013416441325251965, -0.09441108539862886, 0.08190341801437231, -0.08266993890375027, -0.08212763583607363, -0.12960744043818648, 0.10936036565380512, 0.18779273078759634, -0.10492350112492797, 0.020318725332373676, 0.08908385309610427, 0.2027793737515435, -0.24863354350836256, 0.0051988192583482945, 0.05554592465165617, -0.1570232516257982, 0.03166602733141551, 0.010546060302293304, 0.0630951065669457, -0.06754188271454158, 0.03243808832932231, -0.06989731045956905, 0.05119256709824262, 0.0063924092619314634, -0.04714347696226866, -0.15103665537197272], "dtype": "float64"}, {"__nd__": [[0.46588902792416415, 0.3041592260672987, 0.23537879802325878, -0.2279301095063021, -0.13372609307410066, 0.47243624689114494, 0.19475196864547856, -0.2840499784709644, -0.014508535109223975, 0.014120005278032297, -0.08812330218032068, -0.3111339879751542], [0.3485499984799986, 0.14909724368473654, -0.08665848181674095, -0.49922439374406224, -0.20215117793254647, -0.028167221315127926, 0.23379575209009076, -0.13724103615180525, 0.30647662177636037, 0.01633031130291899, -0.24885164818875596, -0.8630824348497694], [-0.37735578840755446, -0.47198549449753124, -0.06999791429918475, 0.07549285741018906, 0.39302185633530967, -0.027543058188027766, -0.3710283876841376, 0.6824365953052164, 0.07034148972380654, -0.10194850575564839, -0.10141642660464378, -0.42813679333400373], [0.1684524668019025, 0.09000943611525193, 0.4547381083105079, 0.3089975982429237, 0.23915268743965454, 0.35638607894973523, -0.1061440358988946, -0.15371818705719584, -0.03522710468081997, -0.3938777822774718, -0.06269794291932947, -0.11832948280064368], [0.1814894043601554, -0.02325327808834935, -0.5911095550750085, -0.009160113627684149, 0.4068337345600061, -0.5683852662240344, -0.24530663494634358, -0.012144572294041632, -0.4148773085013786, -0.18153236374324372, 0.00029399325032743043, -0.41206512586852023], [0.0722911878392889, 0.10707355079424395, -0.39593061803672663, -0.007079346800268197, 0.3090594425276895, -0.06736203002522924, 0.1163666955144383, 0.1789002330118901, -0.08426289737505467, 0.28126979352157344, 0.10221144212077585, -0.31031689584477073], [0.1440086397903481, 0.24710291246117524, 0.22941202419312942, -0.2049193598416035, 0.12649392975828735, 0.40069393420131244, 0.0584608329805849, 0.34492283733226037, 0.3801150281821039, -0.020705882844033987, 0.18869550709604435, -0.09635913943298557], [0.3115075363572964, -0.03592353911914003, -0.1734444962367208, -0.015959633606442217, -0.02985412733632713, 0.10674387254566232, 0.1318926212170228, -0.35696098561758677, -0.20278619512517537, 0.10082514325841625, 0.13740361516433464, 0.019084384609195166], [0.7838063690195227, 0.32979284214159266, -0.028302583340348915, 0.21354057830597753, 0.3968304339978232, 0.10549450214576785, -0.5273887261382306, 0.025906003559772545, 0.012384514323268448, 0.6374296070452246, -0.5533068319841239, 0.4623432504788528], [-0.047974490005538835, 0.27865810146438075, -0.08796721523373785, 0.05267217910221319, 0.339915992454036, 0.02607047603631975, 0.3173669820803876, -0.022869594056347968, -0.12128775687019601, -0.06373620042858327, -0.14875048373194558, -0.03366986223014644], [0.41127840728393567, -0.34816224273271396, -0.4950498810550158, -0.002201297513299809, 0.508986809346989, 0.31787154611182095, 0.4009159885851271, 0.3795231836079876, -0.2141368468197728, 0.04044097986620007, -0.7931109756489297, -0.17747031920519604], [0.18781462674795998, -0.46127260972218187, -0.6121924765646699, -0.38342070666243483, 0.07229697168346216, 0.3574013060255523, 0.783702113101391, 0.4374063673545698, -0.4460916277799128, -0.6869990257780505, 0.05467827962806337, 0.3281180399516352]], "dtype": "float64"}, {"__nd__": [-0.15820544124199443, -0.048443274933231834, -0.040964638472432376, 0.007658707309182861, -0.013500929873532694, 0.056339534711418504, 0.11069068977098791, 0.012307475061205053, 0.08036585593301802, 0.2048350226446136, -0.09651687354068544, -0.09892770997431434], "dtype": "float64"}, 6, {"__nd__": [[false, false, false, false, false, false, true], [false, false, false, false, false, false, false]], "dtype": "bool"}, true], "e": {"__tup__": [{"__nd__": [[[0.37175696688535387, -0.02035896377384728, -0.5948990322208967, 0.7684049571975561, 0.6040657895260895, -0.11035837323465628, -0.1783425735998409, 0.39524280774970266, 0.2557026731364258, 0.6631089822987997, 0.15279031336084173, 0.29511051335059585], [0.05798574910219262, -0.47363404812337184, -0.4986715847028965, 0.9293448441881267, 0.19952552059692413, 0.0035780775320453997, -0.2249151313757864, 0.007337393862715014, -0.747577737079811, 0.724610246954104, -1.3025911369856362, -1.2567954186606014], [0.04375115282924277, -0.0557967776694925, -0.20345217617673486, 0.4887177990941777, 0.2775471245990333, 0.2442191644955104, -0.05537913120315459, 0.16009706703338977, 0.620910741583646, 0.561188000314638, 0.23640932808912124, -0.641469646175452], [-0.11747216164969859, -0.29002099001646553, -0.41845529204361576, 0.12392298787222479, -0.22669514688841733, -0.2791701223971102, 0.09184530745482425, -0.08313450721494893, -0.09669469457450632, 0.4178807899591537, -0.2211237647810344, 0.3724265632327142], [0.40646901165858196, 0.29566789973327745, -1.1611492488567525, 0.13136112442248005, -1.2543096565130933, -0.44025391412617393, 0.6798196624667205, -0.13974955804342382, 0.056937468841016095, 0.6817173497536674, -0.5713061802572257, -0.3993003038201433], [-0.06142446424783703, 0.06363510903636284, -0.7703090165950377, 0.04109939948950457, -0.8533670028590952, -0.09152567597497677, 0.5263763577792849, -0.19353230868129093, 0.07450214776315255, 0.5508310486005864, -0.6832657025390936, -0.826061157926752], [1.7598091014425776e-06, -0.06481342527455076, -0.3621870883042634, 0.2659982433277952, 0.29192558894187143, -0.23824273155356923, -0.23192256329043526, -0.030718542093994904, -0.5048412439741857, 0.6618569490123986, 0.2869067265134817, 1.0485893134453095]], [[-1.5933913384722278, -1.1826191835210171, 1.538493533011903, -0.3111277846755962, 0.7008471059431278, 0.44052581753059905, 0.23538241208312632, -0.22064651261794024, -0.7042477410154078, -0.1261516384252683, 0.16961707790867758, 1.3970404858727945], [-2.4855189724598974, -1.7506150368764706, 1.6720526354021223, -0.9005294298396757, 1.6666803778877228, 0.3381956386546367, -0.9591686888324852, -0.45114591952024513, 0.9425377386418273, -0.6653257700777029, -0.19110895708263642, -0.33925549374615577], [-1.6513688164092986, -2.014450507619447, 1.6679816954219897, 0.13070291331370423, 1.0506380497806411, 0.37841483429601536, -0.24909798870157396, -0.3104094910138622, 1.5432369949286855, -0.028249243474372188, 0.8930426789974548, 0.40189974118722394], [-1.6260529621217135, -1.8736055408678143, 1.7393692776334484, 0.082097299468436, 0.5442861053429131, 0.1028250457167742, -0.056375965928146934, -0.4626070011244103, 0.8505573677169593, -0.1183104381728253, 0.6480297585891748, 0.6142057592900406], [-1.501967924435997, -1.6877371717644276, 1.8881747665985513, 0.15859491577045629, 0.4812465904427351, 0.0653348980502666, -0.005433084266409205, -0.3536590593948567, 0.3047493313192476, -0.18046289905412036, 0.8496796088051493, 1.171736491064806], [-1.9836771895129277, -2.045463124503655, 2.0123010462023347, 0.03140632246070689, 0.5778013600359432, -0.13733425281129316, -0.22015579624375525, -0.686161061518054, 0.12369049945598407, -0.30451148598888933, 0.11360531581100727, 0.5841502963286431], [-1.232889443260194, -1.5651950736758118, 0.5949009730838621, 0.2181822364461237, -0.0071703836552304764, -0.4285887401864248, -0.4206078442580095, -0.2417223554788943, -0.4285178277392141, 0.1440865057812168, 0.25890579786382834, 1.2217869554327518]]], "dtype": "float64"}, {"__nd__": [[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.4636624395719584, 0.5363375604280417, 0.0, 0.0, 0.0, 0.0, 0.0], [0.5342800918499456, 0.2709174125759706, 0.19480249557408388, 0.0, 0.0, 0.0, 0.0], [0.26099099160596834, 0.17180887810734694, 0.2109536755466508, 0.35624645474003397, 0.0, 0.0, 0.0], [0.1792713330318824, 0.16357729396451529, 0.15798566923867907, 0.3131778648425556, 0.18598783892236767, 0.0, 0.0], [0.10202018073508257, 0.17790383058304882, 0.2492061409636116, 0.13333689244736135, 0.18433840148566713, 0.15319455378522848, 0.0], [0.2324082821639315, 0.11773361986103115, 0.23068041526000993, 0.25298962645264966, 0.0684309500809044, 0.09775710618147343, 0.0]], [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.4898827177554357, 0.5101172822445643, 0.0, 0.0, 0.0, 0.0, 0.0], [0.27995461039121045, 0.3795179627660397, 0.3405274268427498, 0.0, 0.0, 0.0, 0.0], [0.25840376129274584, 0.25129179827853904, 0.2751094380675944, 0.21519500236112066, 0.0, 0.0, 0.0], [0.21916439073959812, 0.18869703268805874, 0.1705643723568421, 0.2519286880865457, 0.16964551612895531, 0.0, 0.0], [0.14246300883101295, 0.1760889425850158, 0.18359724435232186, 0.11638118613671589, 0.23873320940248277, 0.14273640869245072, 0.0], [0.0984164067645351, 0.07980173513260715, 0.10713916637556098, 0.0764186059450088, 0.16077247727769073, 0.2512594838893689, 0.2261921246152283]]], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(mha_packed, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")